# Fraud Detection - Exploratory Data Analysis

This notebook performs comprehensive EDA on the fraud detection dataset to understand patterns, relationships, and characteristics of fraudulent vs legitimate transactions.

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Load the dataset
df = pd.read_csv('../data/fraud_data.csv')

# Convert transaction_time to datetime
df['transaction_time'] = pd.to_datetime(df['transaction_time'])

print(f"Dataset Shape: {df.shape}")
print(f"\nDataset Info:")
df.info()

In [ ]:
# Basic statistics
print("Basic Statistics:")
print(df.describe())

# Check for missing values
print("\nMissing Values:")
print(df.isnull().sum())

In [ ]:
# Fraud distribution analysis
fraud_counts = df['is_fraud'].value_counts()
fraud_percentages = df['is_fraud'].value_counts(normalize=True) * 100

print("Fraud Distribution:")
print(f"Legitimate transactions: {fraud_counts[0]} ({fraud_percentages[0]:.2f}%)")
print(f"Fraudulent transactions: {fraud_counts[1]} ({fraud_percentages[1]:.2f}%)")

# Visualize fraud distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Count plot
sns.countplot(data=df, x='is_fraud', ax=ax1)
ax1.set_title('Fraud vs Legitimate Transactions (Count)')
ax1.set_xlabel('Is Fraud (0=No, 1=Yes)')
ax1.set_ylabel('Count')

# Pie chart
ax2.pie(fraud_counts, labels=['Legitimate', 'Fraud'], autopct='%1.1f%%', startangle=90)
ax2.set_title('Fraud Distribution Percentage')

plt.tight_layout()
plt.show()

In [ ]:
# Transaction amount analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Overall amount distribution
sns.histplot(data=df, x='transaction_amount', bins=50, ax=axes[0,0])
axes[0,0].set_title('Overall Transaction Amount Distribution')
axes[0,0].set_xlabel('Transaction Amount ($)')

# Amount distribution by fraud status
sns.histplot(data=df, x='transaction_amount', hue='is_fraud', bins=50, 
             alpha=0.7, ax=axes[0,1])
axes[0,1].set_title('Transaction Amount by Fraud Status')
axes[0,1].set_xlabel('Transaction Amount ($)')
axes[0,1].legend(['Legitimate', 'Fraud'])

# Box plot for amount by fraud status
sns.boxplot(data=df, x='is_fraud', y='transaction_amount', ax=axes[1,0])
axes[1,0].set_title('Transaction Amount Box Plot by Fraud Status')
axes[1,0].set_xlabel('Is Fraud (0=No, 1=Yes)')
axes[1,0].set_ylabel('Transaction Amount ($)')

# Log-transformed amount distribution
df['log_amount'] = np.log1p(df['transaction_amount'])
sns.histplot(data=df, x='log_amount', hue='is_fraud', bins=50, 
             alpha=0.7, ax=axes[1,1])
axes[1,1].set_title('Log-Transformed Transaction Amount by Fraud Status')
axes[1,1].set_xlabel('Log(Transaction Amount + 1)')
axes[1,1].legend(['Legitimate', 'Fraud'])

plt.tight_layout()
plt.show()

# Statistical summary of amounts by fraud status
print("Transaction Amount Statistics by Fraud Status:")
print(df.groupby('is_fraud')['transaction_amount'].describe())

In [ ]:
# Time-based analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Transactions by hour
hourly_fraud = df.groupby('transaction_hour')['is_fraud'].mean() * 100
hourly_count = df.groupby('transaction_hour').size()

ax1 = axes[0,0]
ax1_twin = ax1.twinx()
ax1.bar(hourly_count.index, hourly_count.values, alpha=0.7, color='blue')
ax1_twin.plot(hourly_fraud.index, hourly_fraud.values, 'r-', linewidth=2)
ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('Transaction Count', color='blue')
ax1_twin.set_ylabel('Fraud Rate (%)', color='red')
ax1.set_title('Transaction Count and Fraud Rate by Hour')

# Weekend vs Weekday
weekend_fraud = df.groupby('is_weekend')['is_fraud'].mean() * 100
sns.barplot(x=weekend_fraud.index, y=weekend_fraud.values, ax=axes[0,1])
axes[0,1].set_title('Fraud Rate: Weekend vs Weekday')
axes[0,1].set_xlabel('Is Weekend (0=No, 1=Yes)')
axes[0,1].set_ylabel('Fraud Rate (%)')
axes[0,1].set_xticklabels(['Weekday', 'Weekend'])

# Night time analysis
night_fraud = df.groupby('is_night_time')['is_fraud'].mean() * 100
sns.barplot(x=night_fraud.index, y=night_fraud.values, ax=axes[1,0])
axes[1,0].set_title('Fraud Rate: Night vs Day')
axes[1,0].set_xlabel('Is Night Time (0=No, 1=Yes)')
axes[1,0].set_ylabel('Fraud Rate (%)')
axes[1,0].set_xticklabels(['Day', 'Night'])

# Merchant category analysis
merchant_fraud = df.groupby('merchant_category')['is_fraud'].mean() * 100
merchant_fraud = merchant_fraud.sort_values(ascending=False)
sns.barplot(x=merchant_fraud.values, y=merchant_fraud.index, ax=axes[1,1])
axes[1,1].set_title('Fraud Rate by Merchant Category')
axes[1,1].set_xlabel('Fraud Rate (%)')
axes[1,1].set_ylabel('Merchant Category')

plt.tight_layout()
plt.show()

print("Fraud Rate by Time Periods:")
print(f"Weekend: {weekend_fraud[1]:.2f}% vs Weekday: {weekend_fraud[0]:.2f}%")
print(f"Night: {night_fraud[1]:.2f}% vs Day: {night_fraud[0]:.2f}%")

In [ ]:
# Geographic and device analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Distance from home
sns.boxplot(data=df, x='is_fraud', y='distance_from_home_km', ax=axes[0,0])
axes[0,0].set_title('Distance from Home by Fraud Status')
axes[0,0].set_xlabel('Is Fraud (0=No, 1=Yes)')
axes[0,0].set_ylabel('Distance from Home (km)')
axes[0,0].set_ylim(0, df['distance_from_home_km'].quantile(0.95))  # Limit outliers

# Distance from last transaction
sns.boxplot(data=df, x='is_fraud', y='distance_from_last_transaction_km', ax=axes[0,1])
axes[0,1].set_title('Distance from Last Transaction by Fraud Status')
axes[0,1].set_xlabel('Is Fraud (0=No, 1=Yes)')
axes[0,1].set_ylabel('Distance from Last Transaction (km)')
axes[0,1].set_ylim(0, df['distance_from_last_transaction_km'].quantile(0.95))

# Devices used today
device_fraud = df.groupby('devices_used_today')['is_fraud'].mean() * 100
sns.barplot(x=device_fraud.index, y=device_fraud.values, ax=axes[1,0])
axes[1,0].set_title('Fraud Rate by Number of Devices Used Today')
axes[1,0].set_xlabel('Devices Used Today')
axes[1,0].set_ylabel('Fraud Rate (%)')

# Mobile vs Desktop
mobile_fraud = df.groupby('is_mobile_transaction')['is_fraud'].mean() * 100
sns.barplot(x=mobile_fraud.index, y=mobile_fraud.values, ax=axes[1,1])
axes[1,1].set_title('Fraud Rate: Mobile vs Desktop')
axes[1,1].set_xlabel('Is Mobile (0=Desktop, 1=Mobile)')
axes[1,1].set_ylabel('Fraud Rate (%)')
axes[1,1].set_xticklabels(['Desktop', 'Mobile'])

plt.tight_layout()
plt.show()

In [ ]:
# Customer behavior analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Customer age distribution
sns.histplot(data=df, x='customer_age', hue='is_fraud', bins=20, 
             alpha=0.7, ax=axes[0,0])
axes[0,0].set_title('Customer Age Distribution by Fraud Status')
axes[0,0].set_xlabel('Customer Age')
axes[0,0].legend(['Legitimate', 'Fraud'])

# Customer tenure
sns.histplot(data=df, x='customer_tenure_days', hue='is_fraud', bins=20, 
             alpha=0.7, ax=axes[0,1])
axes[0,1].set_title('Customer Tenure Distribution by Fraud Status')
axes[0,1].set_xlabel('Customer Tenure (Days)')
axes[0,1].legend(['Legitimate', 'Fraud'])

# Ratio to median purchase price
sns.boxplot(data=df, x='is_fraud', y='ratio_to_median_purchase_price', ax=axes[1,0])
axes[1,0].set_title('Ratio to Median Purchase Price by Fraud Status')
axes[1,0].set_xlabel('Is Fraud (0=No, 1=Yes)')
axes[1,0].set_ylabel('Ratio to Median Purchase Price')
axes[1,0].set_ylim(0, df['ratio_to_median_purchase_price'].quantile(0.95))

# Customer transaction count
sns.boxplot(data=df, x='is_fraud', y='customer_transaction_count', ax=axes[1,1])
axes[1,1].set_title('Customer Transaction Count by Fraud Status')
axes[1,1].set_xlabel('Is Fraud (0=No, 1=Yes)')
axes[1,1].set_ylabel('Customer Transaction Count')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation analysis
# Select only numeric columns for correlation
numeric_columns = df.select_dtypes(include=[np.number]).columns
correlation_matrix = df[numeric_columns].corr()

# Plot correlation matrix
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, cmap='coolwarm', center=0,
            square=True, fmt='.2f', cbar_kws={"shrink": .8})
plt.title('Correlation Matrix of Numeric Features')
plt.tight_layout()
plt.show()

# Correlation with fraud
fraud_correlations = correlation_matrix['is_fraud'].sort_values(ascending=False)
print("Correlation with Fraud (sorted):")
print(fraud_correlations)

In [ ]:
# Feature importance analysis using simple statistics
from scipy.stats import chi2_contingency

# Categorical features analysis
categorical_features = ['merchant_category', 'is_weekend', 'is_night_time', 'is_mobile_transaction']

print("Categorical Features Analysis:")
for feature in categorical_features:
    # Create contingency table
    contingency_table = pd.crosstab(df[feature], df['is_fraud'])
    
    # Chi-square test
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    
    print(f"\n{feature}:")
    print(f"  Chi-square statistic: {chi2:.4f}")
    print(f"  P-value: {p_value:.6f}")
    print(f"  Significant: {'Yes' if p_value < 0.05 else 'No'}")
    
    # Show fraud rates by category
    fraud_rates = df.groupby(feature)['is_fraud'].mean() * 100
    print(f"  Fraud rates by category:")
    for category, rate in fraud_rates.items():
        print(f"    {category}: {rate:.2f}%")

In [ ]:
# Key insights summary
print("=" * 50)
print("KEY INSIGHTS FROM EXPLORATORY DATA ANALYSIS")
print("=" * 50)

print("\n1. DATASET OVERVIEW:")
print(f"   - Total transactions: {len(df):,}")
print(f"   - Fraud rate: {df['is_fraud'].mean():.2%}")
print(f"   - Features: {len(df.columns)}")

print("\n2. TRANSACTION PATTERNS:")
legitimate_avg = df[df['is_fraud'] == 0]['transaction_amount'].mean()
fraud_avg = df[df['is_fraud'] == 1]['transaction_amount'].mean()
print(f"   - Average legitimate transaction: ${legitimate_avg:.2f}")
print(f"   - Average fraudulent transaction: ${fraud_avg:.2f}")
print(f"   - Fraud transactions are {fraud_avg/legitimate_avg:.1f}x larger on average")

print("\n3. TIME-BASED PATTERNS:")
high_risk_hours = hourly_fraud.sort_values(ascending=False).head(3)
print(f"   - Highest fraud hours: {list(high_risk_hours.index)} ({', '.join([f'{h}% ' for h in high_risk_hours.values.round(1)])})")
print(f"   - Weekend fraud rate: {weekend_fraud[1]:.2f}% vs Weekday: {weekend_fraud[0]:.2f}%")
print(f"   - Night time fraud rate: {night_fraud[1]:.2f}% vs Day: {night_fraud[0]:.2f}%")

print("\n4. GEOGRAPHIC PATTERNS:")
legitimate_dist = df[df['is_fraud'] == 0]['distance_from_home_km'].median()
fraud_dist = df[df['is_fraud'] == 1]['distance_from_home_km'].median()
print(f"   - Median distance from home (legitimate): {legitimate_dist:.1f} km")
print(f"   - Median distance from home (fraud): {fraud_dist:.1f} km")
print(f"   - Fraud transactions occur {fraud_dist/legitimate_dist:.1f}x farther from home")

print("\n5. DEVICE PATTERNS:")
print(f"   - Mobile fraud rate: {mobile_fraud[1]:.2f}% vs Desktop: {mobile_fraud[0]:.2f}%")
high_device_risk = device_fraud.sort_values(ascending=False).head(1)
print(f"   - Highest risk device count: {high_device_risk.index[0]} devices ({high_device_risk.values[0]:.1f}% fraud rate)")

print("\n6. MERCHANT PATTERNS:")
high_risk_merchant = merchant_fraud.head(1)
print(f"   - Highest risk category: {high_risk_merchant.index[0]} ({high_risk_merchant.values[0]:.1f}% fraud rate)")

print("\n7. BEHAVIORAL INDICATORS:")
legitimate_ratio = df[df['is_fraud'] == 0]['ratio_to_median_purchase_price'].median()
fraud_ratio = df[df['is_fraud'] == 1]['ratio_to_median_purchase_price'].median()
print(f"   - Median ratio to median purchase (legitimate): {legitimate_ratio:.2f}")
print(f"   - Median ratio to median purchase (fraud): {fraud_ratio:.2f}")

print("\n8. TOP CORRELATED FEATURES WITH FRAUD:")
top_correlations = fraud_correlations.abs().sort_values(ascending=False).head(6)
for feature, corr in top_correlations.items():
    if feature != 'is_fraud':
        print(f"   - {feature}: {corr:.3f}")

### Summary of EDA Findings:

1. **Imbalanced Dataset**: Only 2% of transactions are fraudulent, which is typical for fraud detection.

2. **Amount Patterns**: Fraudulent transactions tend to have higher amounts on average.

3. **Time Patterns**: Higher fraud rates during night hours and weekends.

4. **Geographic Patterns**: Fraud transactions occur farther from home and last transaction location.

5. **Device Patterns**: Mobile transactions and multiple device usage show higher fraud rates.

6. **Merchant Categories**: Online and travel categories have higher fraud rates.

7. **Behavioral Indicators**: Unusual spending patterns (ratio to median purchase) are strong fraud indicators.

These insights will guide our feature engineering and model selection in the next phase.